# Notebook 04 - Curadoria de DJ Sets

**Este e o notebook de uso diario.**

Pre-requisitos:
- `.env` configurado (Supabase, Spotify, Gemini)
- Base coletada (pelo menos 10 sets — notebook 01)
- Spotify configurado (para dados acusticos das suas tracks)

> build_set() e sincrono — sem await necessario.

In [ ]:
import sys
sys.path.insert(0, '../')

import os
from dotenv import load_dotenv
from supabase import create_client
from src.enricher.enricher import Enricher
from src.graph.graph import Graph
from src.curator.curator import Curator

load_dotenv('../.env')

sb       = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
enricher = Enricher(sb, os.environ['SPOTIFY_CLIENT_ID'], os.environ['SPOTIFY_CLIENT_SECRET'])
graph    = Graph(sb)
curator  = Curator(sb, enricher, graph, os.environ['GEMINI_API_KEY'])

print('Modulos carregados - pronto!')

## Sua lista de tracks

Formato: `Artista - Titulo`

Tracks fora do Spotify: coloque o nome normalmente. O sistema tentara buscar os dados automaticamente. Se nao encontrar, usa o grafo pelo nome.

In [ ]:
# ════════════════════════════════════════════
# EDITE AQUI antes de gerar a tracklist
# ════════════════════════════════════════════

GENERO = 'goa-psy-trance'  # veja generos disponiveis na ultima celula

MINHAS_TRACKS = [
    'Vini Vici - The Tribe',
    'Astrix - Basic',
    'Simbolica - Vertigo',
    'Neelix - Caustic',
    'Ace Ventura - Superluminal',
    'Liquid Soul - Universe Inside Me',
    'Ticon - Frequencies',
    'GMS - Rockit',
    'Infected Mushroom - Converting Vegetarians',
]

NOME_DO_SET = 'Psytrance Set Marco 2026'

# ════════════════════════════════════════════

print(f'Genero: {GENERO}')
print(f'{len(MINHAS_TRACKS)} tracks na lista')

## Gerar Tracklist

Processa as 4 etapas: enriquecimento -> grafo -> sugestoes -> Gemini.

In [ ]:
resultado = curator.build_set(
    tracks     = MINHAS_TRACKS,
    genre_slug = GENERO,
    list_name  = NOME_DO_SET,
    save_to_db = True,  # salva no historico
)

print('\n' + '='*60)
print(resultado['output_text'])

## Dados acusticos das suas tracks

In [ ]:
import pandas as pd

rows = [{
    'Artista':     t['artist'],
    'Titulo':      t['title'],
    'BPM':         t.get('bpm'),
    'Key':         t.get('camelot_key'),
    'Energy':      t.get('energy'),
    'Fonte':       t.get('source'),
    'Confianca':   t.get('confidence'),
    'Zona tipica': t.get('typical_zone'),
} for t in resultado['tracks_enriched']]

display(pd.DataFrame(rows))

## Aprovar ou rejeitar o set

Um clique. Alimenta o historico para analise futura.

In [ ]:
APROVADO = True  # Mude para False se nao gostou

if resultado.get('proposed_set_id'):
    sb.table('proposed_sets').update({'approved': APROVADO}).eq(
        'id', resultado['proposed_set_id']
    ).execute()
    print(f'{'Aprovado' if APROVADO else 'Rejeitado'} - Set #{resultado["proposed_set_id"]} salvo')
else:
    print('Set nao foi salvo no banco (save_to_db=False)')

## Copiar output para area de transferencia

In [ ]:
try:
    import pyperclip
    pyperclip.copy(resultado['output_text'])
    print('Copiado!')
except ImportError:
    print('Para habilitar: pip install pyperclip')
    print(resultado['output_text'])

## Generos disponiveis

In [ ]:
generos = sb.table('genres').select('slug, name, active').order('name').execute().data
print(f'{"Slug":<35} {"Nome":<30} Ativo')
print('-' * 72)
for g in generos:
    ativo = 'sim' if g['active'] else 'nao'
    print(f"{g['slug']:<35} {g['name']:<30} {ativo}")